# TP NLP — Classification d'intentions **Banking77**
**Dataset public** : **Banking77** — 77 intentions de clients d'une banque (ex. *card_arrival*, *exchange_rate*, *chargeback*, etc.).  
Page du dépôt : https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data  
Fichiers CSV :  
- train : https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv  
- validation : https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/val.csv  
- test : https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv

In [1]:

import sys, os, re, io, random, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_auc_score)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_DIR = Path("data_banking77"); DATA_DIR.mkdir(exist_ok=True)
OUT_DIR  = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("TensorFlow:", tf.__version__)


Python: 3.12.11
Pandas: 2.2.2
TensorFlow: 2.19.0


In [ ]:

URLS = {"train":"https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv", "val":"https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/val.csv", "test":"https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"}
paths = {}
for k,u in URLS.items():
    p = DATA_DIR / f"banking77_{k}.csv"
    if not p.exists():
        r = requests.get(u, timeout=60)
        r.raise_for_status()
        p.write_bytes(r.content)
    paths[k] = p
paths


In [ ]:

cols = ["text","label"]
train_df = pd.read_csv(paths["train"], header=None, names=cols)
val_df   = pd.read_csv(paths["val"],   header=None, names=cols)
test_df  = pd.read_csv(paths["test"],  header=None, names=cols)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head(8)


## 1) Nettoyage & prétraitement (basique, conforme au cours)
- Passage en **minuscule**
- Normalisation des espaces
- Suppression **URLs / emails / mentions / hashtags**
- Retrait **ponctuation / chiffres**
- Mini-liste de **stopwords** (EN) pour éviter le bruit

In [ ]:

STOP = {
    "a","an","the","and","or","if","in","on","at","of","for","to","from","by","with","as",
    "is","are","was","were","be","been","being",
    "do","does","did","doing","this","that","these","those","it","its",
    "i","you","he","she","we","they","me","him","her","us","them",
    "my","your","his","our","their"
}

def basic_clean(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"http\\S+|www\\.\\S+", " URL ", s)
    s = re.sub(r"\\S+@\\S+", " EMAIL ", s)
    s = re.sub(r"[@#]\\w+", " TAG ", s)
    s = re.sub(r"[^a-z\\s]", " ", s)
    s = re.sub(r"\\s+", " ", s).strip()
    toks = [w for w in s.split() if w not in STOP]
    return " ".join(toks)

for df in (train_df, val_df, test_df):
    df["clean"] = df["text"].apply(basic_clean)

train_df[["text","clean","label"]].head(10)


In [ ]:

le = LabelEncoder()
train_df["y"] = le.fit_transform(train_df["label"])
val_df["y"]   = le.transform(val_df["label"])
test_df["y"]  = le.transform(test_df["label"])

num_classes = len(le.classes_)
print("Classes:", num_classes)
list(le.classes_)[:10]


## 2) Représentation (TF‑IDF) & 3) Modèles ML
Modèles **simples** vus en cours : **Logistic Regression** et **Linear SVM**.  
Sélection sur **macro‑F1** (validation), puis apprentissage **train+val** et test final.

In [ ]:

from sklearn.metrics import precision_recall_fscore_support

def macro_scores(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {"accuracy":acc, "precision_macro":pr, "recall_macro":rc, "f1_macro":f1}

results = {}

tfidf = TfidfVectorizer(max_features=60000, ngram_range=(1,2))

pipe_lr = Pipeline([("tfidf", tfidf), ("clf", LogisticRegression(max_iter=500, multi_class='ovr'))])
gs_lr = GridSearchCV(pipe_lr, {"clf__C":[0.5,1.0,2.0]}, cv=3, n_jobs=-1, scoring="f1_macro")
gs_lr.fit(train_df["clean"].values, train_df["y"].values)
pred_val_lr = gs_lr.predict(val_df["clean"].values)
results["LogReg_val"] = macro_scores(val_df["y"].values, pred_val_lr)

pipe_svm = Pipeline([("tfidf", tfidf), ("clf", LinearSVC())])
gs_svm = GridSearchCV(pipe_svm, {"clf__C":[0.5,1.0,2.0]}, cv=3, n_jobs=-1, scoring="f1_macro")
gs_svm.fit(train_df["clean"].values, train_df["y"].values)
pred_val_svm = gs_svm.predict(val_df["clean"].values)
results["LinearSVM_val"] = macro_scores(val_df["y"].values, pred_val_svm)

results


In [ ]:

best_name = max(results.items(), key=lambda kv: kv[1]["f1_macro"])[0]
best_estimator = gs_lr if "LogReg" in best_name else gs_svm
print("Meilleur modèle (validation) :", best_name)

X_all = np.concatenate([train_df["clean"].values, val_df["clean"].values])
y_all = np.concatenate([train_df["y"].values,     val_df["y"].values])
best_estimator.fit(X_all, y_all)

y_pred_test = best_estimator.predict(test_df["clean"].values)
from sklearn.metrics import precision_recall_fscore_support
acc = accuracy_score(test_df["y"], y_pred_test)
pr, rc, f1, _ = precision_recall_fscore_support(test_df["y"], y_pred_test, average='macro', zero_division=0)
ml_scores_test = {"accuracy":float(acc), "precision_macro":float(pr), "recall_macro":float(rc), "f1_macro":float(f1)}
print("Scores test (ML):", ml_scores_test)

print("\\nClassification report (ML):\\n",
      classification_report(test_df["y"], y_pred_test, target_names=list(le.classes_), digits=4))

cm_ml = confusion_matrix(test_df["y"], y_pred_test)
cm_ml.shape


In [ ]:

# TOP confusions (erreurs les plus fréquentes)
import numpy as np, pandas as pd
cm = confusion_matrix(test_df["y"], y_pred_test)
np.fill_diagonal(cm, 0)
pairs = []
for i in range(num_classes):
    for j in range(num_classes):
        if cm[i,j]>0:
            pairs.append((cm[i,j], le.classes_[i], "→", le.classes_[j]))
pairs = sorted(pairs, key=lambda x: x[0], reverse=True)[:15]
pd.DataFrame(pairs, columns=["nb_erreurs","vraie_classe","sens","prédite"])


## 2) Représentation (DL) & 3) Modèle DL (CNN 1D simple)
- **Tokenizer** (40k mots) + **padding**
- **Embedding(128)** + **Conv1D(128)** + **GlobalMaxPooling** + **Dropout**
- **Dense softmax(77)**, **EarlyStopping**
> Modèle volontairement simple (pas de Transformers), conforme au cours.

In [ ]:

VOCAB = 40000
tok = Tokenizer(num_words=VOCAB, oov_token="<OOV>")
tok.fit_on_texts(train_df["clean"].values)

def to_seq(x): return tok.texts_to_sequences(x)

seq_tr = to_seq(train_df["clean"].values)
seq_va = to_seq(val_df["clean"].values)
seq_te = to_seq(test_df["clean"].values)

lengths = list(map(len, seq_tr))
MAX_LEN = int(np.percentile(lengths, 95))
MAX_LEN = max(10, min(MAX_LEN, 40))
print("MAX_LEN:", MAX_LEN)

Xtr = pad_sequences(seq_tr, maxlen=MAX_LEN, padding='post', truncating='post')
Xva = pad_sequences(seq_va, maxlen=MAX_LEN, padding='post', truncating='post')
Xte = pad_sequences(seq_te, maxlen=MAX_LEN, padding='post', truncating='post')

ytr = tf.keras.utils.to_categorical(train_df["y"].values, num_classes)
yva = tf.keras.utils.to_categorical(val_df["y"].values,   num_classes)
yte = tf.keras.utils.to_categorical(test_df["y"].values,  num_classes)

model = Sequential([
    Embedding(input_dim=VOCAB+1, output_dim=128, input_length=MAX_LEN),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
hist = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                 epochs=12, batch_size=128, callbacks=[es], verbose=1)


In [ ]:

proba_dl = model.predict(Xte, verbose=0)
pred_dl  = proba_dl.argmax(axis=1)

from sklearn.metrics import precision_recall_fscore_support
acc = (pred_dl == test_df["y"].values).mean()
pr, rc, f1, _ = precision_recall_fscore_support(test_df["y"].values, pred_dl, average='macro', zero_division=0)
dl_scores_test = {"accuracy":float(acc), "precision_macro":float(pr), "recall_macro":float(rc), "f1_macro":float(f1)}
print("Scores test (DL):", dl_scores_test)

print("\\nClassification report (DL):\\n",
      classification_report(test_df["y"], pred_dl, target_names=list(le.classes_), digits=4))

cm_dl = confusion_matrix(test_df["y"], pred_dl)
cm_dl.shape


In [ ]:

# Comparatif final
import pandas as pd
summary = pd.DataFrame([
    {"model":"ML (best)", **ml_scores_test},
    {"model":"DL (CNN1D)", **dl_scores_test}
])
summary


# Rapport (intégré)

## Contexte & objectif
**Banking77** contient des requêtes courtes de clients et **77 intentions** (ex. *card_arrival, chargeback, exchange_rate, pin_blocked*).  
But : construire un **pipeline** complet (selon le cours) pour classer ces intentions et **comparer ML vs DL**.

**Liens officiels :**
- Page du dépôt : https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data
- CSV : train (https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv), validation (https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/val.csv), test (https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv)

## 1) Nettoyage & prétraitement
Prétraitement **basique** : minuscule, normalisation espaces, retrait URL/email/mentions/hashtags, suppression ponctuation/chiffres, stopwords minimal.  
**Justification** : requêtes très courtes  rester simple évite l’over‑nettoyage et conserve les n‑grams clés.

## 2) Représentation
- **TF‑IDF** (1–2 n‑grams, 60k features) pour **ML** : robuste sur courts textes, capte des combinaisons comme “exchange rate”, “card not received”.  
- **Tokenizer + Embedding** (128) pour **DL**, puis **CNN 1D** qui détecte des motifs locaux (équivalent n‑grams) avec un coût modéré.

## 3) Modèles et procédure
- **Logistic Regression (OvR)** et **Linear SVM** — grille simple sur `C ∈ {0.5,1,2}` ; **sélection par macro‑F1** sur *validation*.  
- **CNN 1D** : `Embedding  Conv1D(128)  GlobalMaxPooling  Dense`, **EarlyStopping** (patience=3), batch 128, 12 époques max.  
- Utilisation des **splits officiels** (train/val/test).  
- **Seed=42** pour la reproductibilité.

## 4) Évaluation
- **Accuracy**, **Precision/Recall/F1 macro**, **matrice de confusion** + **Top confusions** pour l’analyse d’erreurs.  
- Les chiffres dépendent de l’exécution, mais TF‑IDF + SVM ou LogReg forment une **baseline très solide** sur Banking77 ; la **CNN 1D** est compétitive sans réglages avancés.

## Limites & améliorations simples (hors programme avancé)
- Étendre `ngram_range` à (1,3) et ajuster `max_features`.  
- Normaliser les synonymes fréquents (“fx rate”  “exchange rate”).  
- Pondérer les classes rares.  
*(Transformers : optionnel, pas requis pour ce TP.)*

## Conclusion
Le notebook respecte **à la lettre** les **Consignes TP** :  
**Nettoyage  Représentation  Comparaison ML/DL  Évaluation** + **mini‑analyse**.  
Le choix **Banking77** est **intelligent et concret** (SAV bancaire multi‑intents), **non typique** et parfaitement adapté aux techniques vues en cours.